In [1]:
import os
import sqlite3
import pandas as pd

In [2]:
print(os.getcwd())

c:\Users\salit\OneDrive\Escritorio\TB\GitHub\whiteberry\parte1


In [3]:
connection = sqlite3.connect("sql-murder-mystery.db")

cursor = connection.cursor()

In [4]:
##  to find the names of the tables in this database
pd.read_sql("""
    SELECT name 
    FROM sqlite_master
    WHERE type = 'table'
""", connection)

,name
0,crime_scene_report
1,drivers_license
2,person
3,facebook_event_checkin
4,interview
5,get_fit_now_member
6,get_fit_now_check_in
7,income
8,solution


In [5]:
# find the structure of the `crime_scene_report` table
cursor.execute("""
    SELECT sql 
    FROM sqlite_master
    WHERE name = 'crime_scene_report'
""")

print(cursor.fetchone())

('CREATE TABLE crime_scene_report (\n        date integer,\n        type text,\n        description text,\n        city text\n    )',)


In [6]:
# 1 query inicial pra buscar crímenes cometidos el 15 de enero de 2018 en sql City
cursor.execute("""
SELECT * FROM crime_scene_report WHERE date = 20180115  AND type = 'murder' AND city = 'SQL City'""")


# para que muestre la descripción entera, si no se corta
pd.set_option('display.max_colwidth', None)
df_1

df_1 = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
df_1


NameError: name 'df_1' is not defined

In [ ]:
# 2 Persona que vive en el último número de Northwestern Dr
cursor.execute("""
SELECT *  
FROM person p
JOIN (
    SELECT MAX(address_number) AS max_val
    FROM person
    WHERE address_street_name = 'Northwestern Dr'
) x
ON p.address_number = x.max_val
WHERE p.address_street_name = 'Northwestern Dr'""")

df_2 = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
df_2
### id person = 14887

,id,name,license_id,address_number,address_street_name,ssn,max_val
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949,4919


In [ ]:
#3  Personas que viven en el último número de Northwestern Dr y en Franklin Ave
cursor.execute("""
SELECT * FROM person WHERE address_street_name LIKE "%Franklin Ave%" OR id = 14887""")

df_3 = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
df_3


,id,name,license_id,address_number,address_street_name,ssn
0,12207,Wilmer Wolever,509484,139,Franklin Ave,636825374
1,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949
2,16371,Annabel Miller,490173,103,Franklin Ave,318771143
3,17683,Johnnie Schee,968887,1277,Franklin Ave,815977821
4,18651,Carleen Etoll,356746,22,Franklin Ave,193369255
5,22636,Zachary Ybarbo,768359,785,Franklin Ave,285346605
6,24737,Gema Nantz,273410,3968,Franklin Ave,180545802
7,30654,Clarita Rickels,418084,2254,Franklin Ave,714941023
8,32264,Shelby Dezeeuw,735415,1391,Franklin Ave,143197463
9,33793,Amado Mattan,161915,99,Franklin Ave,125205748


In [ ]:
# 4 Entrevistas de los testigos posibles testigos (personas cuyo id están entre las personas que viven n el último número de Northwestern Dr y en Franklin Ave)
cursor.execute("""
SELECT * FROM interview WHERE person_id IN (SELECT id FROM person WHERE address_street_name LIKE "%Franklin Ave%" OR id = 14887)""")
df_4 = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
df_4

,person_id,transcript
0,33793,wood--(she considered him to be a footman because he was in livery:\n
1,88966,like them raw.’\n
2,67292,\n
3,62292,\n
4,83003,"An enormous puppy was looking down at her with large round eyes, and\n"
5,95119,"So she sat on, with closed eyes, and half believed herself in\n"
6,61001,with a smile. There was a dead silence.\n
7,22636,\n
8,60944,"Lastly, she pictured to herself how this same little sister of hers\n"
9,84531,"way?’, holding her hand on the top of her head to feel which way it was\n"


In [ ]:
# 5 entrevistas a testigos junto con datos de los testigos que viven en el último número de farankiln ave or ultimo numero de Northwestern dr
cursor.execute("""
SELECT i.*, p.* FROM interview i
left join person p
on i.person_id = p.id
WHERE i.person_id IN (SELECT id FROM person WHERE address_street_name LIKE "%Franklin Ave%" OR id = 14887)""")
df_5 = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
df_5

# 16371

,person_id,transcript,id,name,license_id,address_number,address_street_name,ssn
0,33793,wood--(she considered him to be a footman because he was in livery:\n,33793,Amado Mattan,161915,99,Franklin Ave,125205748
1,88966,like them raw.’\n,88966,Raul Eads,570927,2066,Franklin Ave,764132739
2,67292,\n,67292,Renita Roperto,801041,625,Franklin Ave,894442480
3,62292,\n,62292,Isaiah Holsten,913856,747,Franklin Ave,995368430
4,83003,"An enormous puppy was looking down at her with large round eyes, and\n",83003,Wilmer Casella,672050,3564,Franklin Ave,569057540
5,95119,"So she sat on, with closed eyes, and half believed herself in\n",95119,Hong Lisa,825828,375,Franklin Ave,113438176
6,61001,with a smile. There was a dead silence.\n,61001,Laurine Bousman,197150,247,Franklin Ave,431360364
7,22636,\n,22636,Zachary Ybarbo,768359,785,Franklin Ave,285346605
8,60944,"Lastly, she pictured to herself how this same little sister of hers\n",60944,Ricki Bidding,443364,2072,Franklin Ave,614976330
9,84531,"way?’, holding her hand on the top of her head to feel which way it was\n",84531,Edgar Mendieta,315790,3881,Franklin Ave,458847132


In [ ]:
# 7 eventos el día del crimen en los que el asistente no era testigo
cursor.execute("""
SELECT f.*, p.* FROM facebook_event_checkin f
LEFT JOIN person p
ON p.id = f.person_id  
WHERE f.event_id = 4719
AND f.date = 20180115
AND p.id NOT IN(SELECT id FROM person WHERE address_street_name LIKE "%Franklin Ave%" OR id = 14887)""")

df_6 = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
df_6
# persons id tres 14887(testigo), 16371 (es el segundo testigo deducible por la dirección ) 67318 Jeremy Bowers el malo

,person_id,event_id,event_name,date,id,name,license_id,address_number,address_street_name,ssn
0,67318,4719,The Funky Grooves Tour,20180115,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279


In [ ]:
## check the solution 

cursor.execute("INSERT INTO solution VALUES (1, 'Jeremy Bowers')")
connection.commit()

In [ ]:
cursor.execute("SELECT value FROM solution")
pd.DataFrame(cursor.fetchall(), columns=["value"])

,value
0,"Congrats, you found the murderer! But wait, there's more... If you think you're up for a challenge, try querying the interview transcript of the murderer to find the real villain behind this crime. If you feel especially confident in your SQL skills, try to complete this final step with no more than 2 queries. Use this same INSERT statement with your new suspect to check your answer."


In [ ]:
# la transcripción del primer sospechoso

cursor.execute("""
SELECT transcript FROM interview 
WHERE person_id = 67318
""")

df_7 = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
print(df_7['transcript'][0])

I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.



In [ ]:
cursor.execute("""
SELECT p.name, p.id
FROM person p
JOIN drivers_license dl ON p.license_id = dl.id
JOIN facebook_event_checkin fb ON p.id = fb.person_id
WHERE dl.hair_color = 'red'
AND dl.gender = 'female'
AND dl.height BETWEEN 65 AND 67
AND dl.car_make = 'Tesla'
AND dl.car_model = 'Model S'
AND fb.event_name = 'SQL Symphony Concert'
AND fb.date BETWEEN 20171201 AND 20171231
GROUP BY p.id
HAVING COUNT(fb.event_name) = 3
""")

df_villain = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
df_villain

,name,id
0,Miranda Priestly,99716


In [ ]:
## check the solution v2

cursor.execute("INSERT INTO solution VALUES (1, 'Miranda Priestly')")
connection.commit()

In [ ]:
cursor.execute("SELECT value FROM solution")
pd.DataFrame(cursor.fetchall(), columns=["value"])

,value
0,"Congrats, you found the brains behind the murder! Everyone in SQL City hails you as the greatest SQL detective of all time. Time to break out the champagne!"


**El asesino material del crimen es Jeremy Bowers**  
**La autora intelectual del crimen es Miranda Priestly**